In [76]:
import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

In [77]:
class GridWorld:
    def __init__(self):
        self.gridworld = [[0] * 9 for i in range(6)]

        self.gridworld[1][2] = 2
        self.gridworld[2][2] = 2
        self.gridworld[3][2] = 2

        self.gridworld[0][7] = 2
        self.gridworld[1][7] = 2
        self.gridworld[2][7] = 2

        self.gridworld[4][5] = 2

        self.gridworld[2][0] = 1

        self.agent_location_x = 2
        self.agent_location_y = 0

    def action_space(self):
        return [0, 1, 2, 3]

    def step(self, a):
        if a == 0:
            self.agent_location_y_temp = max(self.agent_location_y - 1, 0)

            if self.gridworld[self.agent_location_x][self.agent_location_y_temp] == 2:
                return self.agent_location_x, self.agent_location_y

            self.gridworld[self.agent_location_x][self.agent_location_y] = 0
            self.agent_location_y = self.agent_location_y_temp
            self.gridworld[self.agent_location_x][self.agent_location_y] = 1

        if a == 1:
            self.agent_location_x_temp = min(self.agent_location_x + 1, 5)

            if self.gridworld[self.agent_location_x_temp][self.agent_location_y] == 2:
                return self.agent_location_x, self.agent_location_y

            self.gridworld[self.agent_location_x][self.agent_location_y] = 0
            self.agent_location_x = self.agent_location_x_temp
            self.gridworld[self.agent_location_x][self.agent_location_y] = 1

        if a == 2:
            self.agent_location_y_temp = min(self.agent_location_y + 1, 8)

            if self.gridworld[self.agent_location_x][self.agent_location_y_temp] == 2:
                return self.agent_location_x, self.agent_location_y

            self.gridworld[self.agent_location_x][self.agent_location_y] = 0
            self.agent_location_y = self.agent_location_y_temp
            self.gridworld[self.agent_location_x][self.agent_location_y] = 1

        if a == 3:
            self.agent_location_x_temp = max(self.agent_location_x - 1, 0)

            if self.gridworld[self.agent_location_x_temp][self.agent_location_y] == 2:
                return self.agent_location_x, self.agent_location_y

            self.gridworld[self.agent_location_x][self.agent_location_y] = 0
            self.agent_location_x = self.agent_location_x_temp
            self.gridworld[self.agent_location_x][self.agent_location_y] = 1

        return self.agent_location_x, self.agent_location_y

    def reset(self):
        self.gridworld[1][2] = 2
        self.gridworld[2][2] = 2
        self.gridworld[3][2] = 2

        self.gridworld[0][7] = 2
        self.gridworld[1][7] = 2
        self.gridworld[2][7] = 2

        self.gridworld[4][5] = 2

        self.gridworld[2][0] = 1

        self.agent_location_x = 2
        self.agent_location_y = 0

In [78]:
env = GridWorld()

In [79]:
def one_hot_encoding(action):
    if action == 0:
        return [1, 0, 0, 0]
    elif action == 1:
        return [0, 1, 0, 0]
    elif action == 2:
        return [0, 0, 1, 0]
    elif action == 3:
        return [0, 0, 0, 1]

In [80]:
def do_n_samples(n):

    data_X = []
    data_Y = []

    for i in range(n):
        previous_state = env.agent_location_x, env.agent_location_y

        action = np.random.randint(4)
        next_state = env.step(action)


        input = list(previous_state) + one_hot_encoding(action)

        output = list(next_state)

        data_X.append(input)

        data_Y.append(output)

    return data_X, data_Y

Writing out the program for intrinsic motivation.

Essentially, there are a few things here:

1. We need to compute an inverse dynamics model which is basically computes the action taken given a particular starting state and a particular ending state
2. We need to compute a model that would predict an particular ending state given a starting state and an action.

It need to be incredibly good at computing these two objectives. A good starting point is just a simple linear regression. We take the state, the action and the output needs to be the ending state.


A reasonable model for this would be a neural network with no activation function and two outputs. For the x and y of this.

I think the action could be better represented in a one hot encoding format than in a numerical format.

For the inverse dynamics model, the output could be coupled with a sigmoid function and a softmax function to produce probabilities of things.

The forward dynamics model is primarily a regression model and the inverse dynamics model is primarily a classification based neural network.

In [85]:
class ForwardDynamicsModel(nn.Module):
    def __init__(self):
        super(ForwardDynamicsModel, self).__init__()
        # 3 inputs -> 2 outputs
        self.linear = nn.Linear(in_features=6, out_features=2, bias=True)

    def forward(self, x):
        return self.linear(x)

In [ ]:
# x = torch.tensor(data_X).float()
# y = torch.tensor(data_Y).float()


In [86]:
model = ForwardDynamicsModel()
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.000001)


# predicted_output = model(x)

# loss = criterion(y, predicted_output)
# loss.backward()
# optimizer.step()

In [92]:
for i in tqdm(range(1000000)):

    data_X, data_Y = do_n_samples(5)

    x = torch.tensor(data_X).float()
    y = torch.tensor(data_Y).float()

    predicted_output = model(x)

    loss = criterion(y, predicted_output)
    loss.backward()
    optimizer.step()

100%|██████████| 1000000/1000000 [08:22<00:00, 1988.28it/s]


In [93]:
predicted_output

tensor([[-1.2202e+27, -8.8769e+27],
        [-1.6586e+27, -1.2066e+28],
        [-1.2202e+27, -8.8769e+27],
        [-1.4990e+27, -1.0905e+28],
        [-1.7070e+27, -1.2418e+28]], grad_fn=<AddmmBackward0>)

In [91]:
for name, param in model.named_parameters():
    print(f"{name} => shape: {param.shape}")
    print(param.data)

linear.weight => shape: torch.Size([2, 6])
tensor([[  203.4489,  -207.9374,   -77.9362,    95.0674,  -163.1990,  -194.3409],
        [ 1201.2119, -1350.8917,  -683.8600,   741.9749, -1237.3027, -1553.4213]])
linear.bias => shape: torch.Size([2])
tensor([ -340.9603, -2732.8132])


In [88]:
loss

tensor(47724716., grad_fn=<MseLossBackward0>)

In [17]:
predicted_output

tensor([[-0.5709, -0.0083],
        [-0.5835,  0.2408],
        [-0.5709, -0.0083],
        [-0.9412, -0.0119],
        [-0.6612, -0.0387]], grad_fn=<AddmmBackward0>)

In [18]:
output = torch.tensor(data_Y).float()

In [19]:
output

tensor([[3., 0.],
        [2., 0.],
        [3., 0.],
        [3., 1.],
        [2., 1.]])

In [ ]:
# y = torch.randn(5, 2)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# Training step
# optimizer.zero_grad()
# predictions = model(x)
loss = criterion(output, predicted_output)

# # Computing Intrinsic motivation
# η = 5  # How much intrinsic motivation should be valued
# intrinsic_motivation_reward = η * loss

loss.backward()
optimizer.step()


In [6]:
class InverseDynamicsModel(nn.Module):
    def __init__(self):
        super(InverseDynamicsModel, self).__init__()
        # No hidden layer: direct linear from 4 inputs to 4 outputs
        self.linear = nn.Linear(4, 4)

    def forward(self, x):
        x = self.linear(x)  # Linear transformation
        x = torch.sigmoid(x)  # Sigmoid applied elementwise
        x = F.softmax(x, dim=1)  # Softmax over the 4 outputs (per sample)
        return x

We also need to add intrinsic motivation to the mix. Essentially, intrinsic motivation would be the loss of the forward dynamics model. Essentially, it would be the difference in prediction for the next state and the actual next state. This would be multiplied by eta to compute intrinsic motivation.

In [7]:
# Instantiate the model
model = ForwardDynamicsModel()

# Example input: batch of 5 samples, each with 3 features
x = torch.randn(5, 6)  # (batch_size, input_features)

# Forward pass
output = model(x)

print("Output:\n", output)

Output:
 tensor([[-0.9010, -0.1215],
        [ 0.1507, -0.1843],
        [-1.3187,  0.0800],
        [-0.3682,  0.0784],
        [-0.8440, -0.3946]], grad_fn=<AddmmBackward0>)


In [8]:
# Dummy targets for 5 samples, 2 outputs each
y = torch.randn(5, 2)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# Training step
optimizer.zero_grad()
predictions = model(x)
loss = criterion(predictions, y)

# Computing Intrinsic motivation
η = 5 # How much intrinsic motivation should be valued
intrinsic_motivation_reward = η * loss

loss.backward()
optimizer.step()

print("Loss:", loss.item())

Loss: 1.02359938621521


Basically, we want to optimize things in a way that would find the parameters for lowering this particular loss.

The loss function would occur as follows:

-lambda * intrinsic_motivation_reward + (1-beta) * inverse_model_loss + (beta) * forward_model_loss 

We would like to find the parameters that would minimize this loss.

Henceforth, the backpropogation must also happen in a similar way. It needs to combine the individual losses and intrinsic motivation and aim to find the derivatives that would minimize that loss function. 

For starters we can try to train them separately and thenn combine them in some way.

What would the policy look like?

The policy should look to maximize intrinsic motivation.

The policy will be a NN similar to the inverse dynamics model. It will take in the current state and output the action. It will have sigmoidal function coupled with a softmax function.. 

In [9]:
class PolicyNetwork(nn.Module):
    def __init__(self):
        super(PolicyNetwork, self).__init__()
        # No hidden layer: direct linear from 4 inputs to 4 outputs
        self.linear = nn.Linear(2, 4)

    def forward(self, x):
        x = self.linear(x)  # Linear transformation
        x = torch.sigmoid(x)  # Sigmoid applied elementwise
        x = F.softmax(x, dim=1)  # Softmax over the 4 outputs (per sample)
        return x

The policy network's aim would be to maximize intrinsic motivation. Hence, it would be perhaps a gradient ascent style thing where the derivatives computed would be about the intrinsic motivation and trying to maximize that. 

Let's start with something simple and train the forward dynamics model with a random policy to generate the data just for some practice and then we can scale things up from that.